# 📊 Evaluación de Modelos — FIDE Chess Dataset

**Evaluación 2 — Validación y Evaluación (20%)**

Este notebook cubre:
1. Validación cruzada robusta (5-fold)
2. Métricas múltiples (Accuracy, Precision, Recall, F1, ROC-AUC)
3. Matrices de confusión visuales
4. Curvas ROC
5. Comparación gráfica avanzada entre modelos

In [ ]:
%load_ext kedro.ipython

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, RocCurveDisplay,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42

In [ ]:
# Cargar y preparar datos
df = catalog.load('fide_preprocessed_data')

FEATURE_COLS = ['rating_std_avg', 'rating_change', 'total_months_active',
                'age_approx', 'gender_encoded', 'title_encoded']
TARGET = 'is_expert'

available = [c for c in FEATURE_COLS if c in df.columns]
ml_data = df[available + [TARGET]].dropna()

X = ml_data[available]
y = ml_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 1. Validación Cruzada (5-Fold)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=RANDOM_STATE, probability=True),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_results = {}

for name, model in models.items():
    print(f'Cross-validating {name}...')
    cv = cross_validate(model, X_train, y_train, cv=5, scoring=scoring, return_train_score=False)
    cv_results[name] = {metric: cv[f'test_{metric}'] for metric in scoring}

# Tabla de resultados
cv_summary = pd.DataFrame({
    name: {metric: f"{np.mean(scores):.4f} \u00b1 {np.std(scores):.4f}"
           for metric, scores in metrics.items()}
    for name, metrics in cv_results.items()
}).T

display(cv_summary)

In [ ]:
# Boxplots de la distribución de scores en cross-validation
fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, metric in zip(axes, scoring):
    data = [cv_results[name][metric] for name in models]
    bp = ax.boxplot(data, labels=list(models.keys()), patch_artist=True)
    colors = sns.color_palette('viridis', len(models))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax.set_title(metric.upper())
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Distribución de Métricas en Cross-Validation (5-Fold)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Experto', 'Experto']).plot(ax=ax, cmap='Blues')
    ax.set_title(name, fontsize=10)

plt.suptitle('Matrices de Confusión por Modelo', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 3. Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

colors = sns.color_palette('viridis', len(models))

for (name, model), color in zip(models.items(), colors):
    model.fit(X_train, y_train)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC \u2014 Comparación de Modelos')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Tabla Final de Métricas en Test

In [ ]:
final_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    }
    if y_proba is not None:
        metrics['ROC-AUC'] = roc_auc_score(y_test, y_proba)
    final_results[name] = metrics

results_df = pd.DataFrame(final_results).T.sort_values('F1-Score', ascending=False)
display(results_df.style.format('{:.4f}').background_gradient(cmap='YlGn'))

## 5. Conclusiones de la Evaluación

- La validación cruzada 5-fold confirma la estabilidad de los modelos.
- Las curvas ROC permiten evaluar el trade-off entre sensibilidad y especificidad.
- Las matrices de confusión revelan si los modelos tienen sesgo hacia alguna clase.
- El siguiente paso es optimizar los hiperparámetros del mejor modelo.